# 03. Campanha de fintech: dois fatores de uma vez (fatorial 2x2)

**Setor:** fintech (CRM). **Decisão:** como desenhar a campanha de ativação?
Dois fatores binários entram em jogo ao mesmo tempo:

- **cashback** (oferecer vs. não), fator `A`
- **horário de envio** (manhã vs. noite), fator `B`

Em vez de testar um por vez, um desenho **fatorial 2x2** mede, num único
experimento, os dois **efeitos principais** e a **interação** (será que cashback
rende mais quando enviado à noite?). Base teórica:
[II. Desenhos](../../../docs/pt-br/teoria/02-desenhos.md) (tópico 7) e
[III. Estimação](../../../docs/pt-br/teoria/03-estimacao.md) (tópico 12).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "fintech_crm.csv")
print(df.shape)
df.head()


## 1. Desenho fatorial e o resultado

`FactorialDesign(factors=["cashback", "send_time"], n_per_cell=1000)` sorteia as
`2^2 = 4` células com o mesmo tamanho. O resultado (ativações) é gerado a partir
dos fatores atribuídos, com coeficientes conhecidos numa codificação `{0,1}`:

```
ativacoes = 30 + 4*cashback + 2*send_time + 3*(cashback*send_time) + ruido
```

Repare que a codificação `{0,1}` da simulação **não** é a mesma escala dos
contrastes `±1` que o estimador usa. A teoria (tópico 12) mostra que o efeito
principal estimado é `b + metade da interação`. Então esperamos recuperar:

- cashback: `4 + 3/2 = 5,5`
- send_time: `2 + 3/2 = 3,5`
- interação: `3/2 = 1,5`


In [ ]:
from skxperiments.core.assignment import FactorialAssignment
from skxperiments.design.factorial import FactorialDesign

design = FactorialDesign(factors=["cashback", "send_time"], n_per_cell=1000, seed=303)
assignment = design.randomize(df[["customer_id", "tenure_months"]].copy())

A = assignment.data_["cashback"].to_numpy()
B = assignment.data_["send_time"].to_numpy()
data = assignment.data_.copy()
data["activations"] = 30 + 4 * A + 2 * B + 3 * (A * B) + df["noise"].to_numpy()

assignment = FactorialAssignment(
    data=data, design=design, factor_cols=assignment.factor_cols,
    cell_sizes=assignment.cell_sizes_, seed=303,
)
assignment.data_.groupby(["cashback", "send_time"])["activations"].mean().round(2)


In [ ]:
from skxperiments.estimators.factorial_estimator import FactorialEstimator

result = FactorialEstimator(outcome_col="activations").fit(assignment).estimate()
expected = {"cashback": 5.5, "send_time": 3.5, "cashback:send_time": 1.5}
rows = []
for key, value in result.effects.items():
    name = ":".join(key)
    rows.append({"effect": name, "estimated": round(value, 3), "expected": expected[name]})
pd.DataFrame(rows)


In [ ]:
from skxperiments.reporting import plot_interaction

ax = plot_interaction(result)
ax.set_title("Main effects and interaction")
ax.figure


## Decisão

Os três efeitos são recuperados perto do esperado (`5,5`, `3,5`, `1,5`). A
**interação positiva** confirma que cashback e envio à noite se reforçam: juntos
rendem mais do que a soma dos efeitos isolados sugeriria. A recomendação de
campanha é combinar os dois fatores no nível alto. Um desenho "um fator por vez"
jamais revelaria essa sinergia.

Próximo passo: [04. Rerandomização na logística](04_logistics_rerandomization.ipynb).
